# Diabetes Risk Assessment

**SciEncephalon AI · Summer Intern Series 2026**  
**Created by Reagan Lundy**

This final notebook documents the data, calibrated model, test-set metrics,
threshold trade-off, calibration, lift/gain analysis, subgroup audit, and safe
lifestyle coach used by the public Streamlit app.

> **Not medical advice.** This educational model is not clinically validated,
> cannot diagnose diabetes, and must not be used for medical decisions.

## 1. Setup

The notebook imports the same project functions used by the app so the reported
calculations and the public interface stay consistent.

In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, brier_score_loss, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split

from data.loader import load_pima, load_synthetic, load_synthetic_brfss
from src.fairness import subgroup_metrics
from src.lifestyle_coach import llm_compose
from src.pipeline import (
    calibration_curve,
    lift_gain_table,
    predict_proba,
    threshold_sweep,
    train,
)

SEED = 42
np.random.seed(SEED)
print("Setup complete. Random seed:", SEED)

Setup complete. Random seed: 42


## 2. Reproducible data

The offline baseline is synthetic so this notebook always runs. The PIMA loader
uses real public data when reachable and clearly labels a synthetic fallback.
PIMA's impossible zero measurements are median-imputed when real data loads.

In [2]:
synthetic_brfss = load_synthetic_brfss(n=10_000, seed=SEED)
pima = load_pima(clean=True)

pd.DataFrame([
    {"dataset": "Synthetic BRFSS baseline", "rows": len(synthetic_brfss), "source": synthetic_brfss.attrs.get("source")},
    {"dataset": "PIMA comparison", "rows": len(pima), "source": pima.attrs.get("source")},
])

[loader] Could not load Pima from https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv (URLError); falling back to synthetic.


                    dataset   rows              source
0  Synthetic BRFSS baseline  10000     synthetic_brfss
1           PIMA comparison    500  synthetic_fallback

## 3. Train once and evaluate on held-out data

Each result below uses a stratified 75/25 train/test split. The test rows are not
used to fit the model. Sensitivity and specificity use a 0.50 threshold.

In [3]:
def evaluate(df, label):
    X = df.drop(columns=["outcome"])
    y = df["outcome"].to_numpy()
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, stratify=y, random_state=SEED
    )
    model = train(X_train, y_train, seed=SEED)
    probability = predict_proba(model, X_test)
    prediction = (probability >= 0.50).astype(int)
    tn = int(((prediction == 0) & (y_test == 0)).sum())
    fp = int(((prediction == 1) & (y_test == 0)).sum())
    return {
        "label": label, "model": model, "X_test": X_test, "y_test": y_test,
        "probability": probability,
        "auc": roc_auc_score(y_test, probability),
        "accuracy": accuracy_score(y_test, prediction),
        "sensitivity": recall_score(y_test, prediction, zero_division=0),
        "specificity": tn / (tn + fp) if tn + fp else 0.0,
        "brier": brier_score_loss(y_test, probability),
    }

brfss_result = evaluate(synthetic_brfss, "Synthetic BRFSS baseline")
pima_result = evaluate(pima, "PIMA comparison")

metrics = pd.DataFrame([
    {key: result[key] for key in ["label", "auc", "accuracy", "sensitivity", "specificity", "brier"]}
    for result in [brfss_result, pima_result]
])
metrics.round(3)

                      label    auc  accuracy  sensitivity  specificity  brier
0  Synthetic BRFSS baseline  0.720     0.681        0.372        0.865  0.202
1           PIMA comparison  0.714     0.784        0.133        0.989  0.163

**Interpretation:** AUC measures ranking,
sensitivity measures how many true cases are caught, specificity measures how
many non-cases are left unflagged, and Brier score measures probability error.
The data-source table must be checked before interpreting the PIMA row because a
network failure causes an explicitly labeled synthetic fallback.

## 4. Calibration and threshold trade-off

Calibration compares predicted probabilities with observed outcome rates.
Changing the threshold changes the high-risk label, not the predicted probability.

In [4]:
centers, observed, expected = calibration_curve(
    brfss_result["y_test"], brfss_result["probability"], n_bins=10
)
calibration_table = pd.DataFrame({
    "average predicted probability": centers,
    "observed diabetes rate": observed,
    "perfect calibration reference": expected,
})
calibration_table.round(3)

   average predicted probability  ...  perfect calibration reference
0                          0.068  ...                          0.068
1                          0.173  ...                          0.173
2                          0.248  ...                          0.248
3                          0.350  ...                          0.350
4                          0.451  ...                          0.451
5                          0.543  ...                          0.543
6                          0.642  ...                          0.642
7                          0.741  ...                          0.741
8                          0.842  ...                          0.842
9                          0.936  ...                          0.936

[10 rows x 3 columns]

In [5]:
thresholds = threshold_sweep(brfss_result["y_test"], brfss_result["probability"])
thresholds.round(3)

    threshold  sensitivity  specificity     f1
0        0.05        1.000        0.002  0.544
1        0.10        0.999        0.010  0.546
2        0.15        0.990        0.042  0.551
3        0.20        0.951        0.217  0.583
4        0.25        0.892        0.374  0.606
5        0.30        0.805        0.498  0.608
6        0.35        0.700        0.616  0.598
7        0.40        0.592        0.721  0.575
8        0.45        0.496        0.792  0.538
9        0.50        0.372        0.865  0.465
10       0.55        0.250        0.936  0.368
11       0.60        0.168        0.967  0.275
12       0.65        0.105        0.983  0.185
13       0.70        0.060        0.992  0.112
14       0.75        0.029        0.996  0.056
15       0.80        0.015        0.997  0.029
16       0.85        0.012        0.999  0.023
17       0.90        0.006        0.999  0.013
18       0.95        0.002        1.000  0.004

## 5. Lift and gain with ten deciles

People are ranked from highest to lowest predicted risk. Gain is the share of all
diabetes cases captured by the targeted group. Lift equals gain divided by the
share of the population targeted; 2.0x means twice the capture expected from
random selection.

In [6]:
lift_gain = lift_gain_table(
    brfss_result["y_test"], brfss_result["probability"], n_deciles=10
)
lift_gain[[
    "target_population_pct", "diabetic_patients_identified_pct", "lift",
    "diabetic_patients_found", "targeted_people"
]]

   target_population_pct  ...  targeted_people
0                   10.0  ...              250
1                   20.0  ...              500
2                   30.0  ...              750
3                   40.0  ...             1000
4                   50.0  ...             1250
5                   60.0  ...             1500
6                   70.0  ...             1750
7                   80.0  ...             2000
8                   90.0  ...             2250
9                  100.0  ...             2500

[10 rows x 5 columns]

## 6. Subgroup performance audit

These tables compare performance across BMI and age groups. A sensitivity gap is
the highest subgroup sensitivity minus the lowest. Lower sensitivity means more
true diabetes cases are missed in that subgroup at the same threshold.

In [7]:
bmi_audit = subgroup_metrics(
    brfss_result["model"], brfss_result["X_test"], brfss_result["y_test"], by="bmi_bucket"
)
age_audit = subgroup_metrics(
    brfss_result["model"], brfss_result["X_test"], brfss_result["y_test"], by="age_group"
)
print("BMI groups")
print(bmi_audit.round(3).to_string(index=False))
print("Age groups")
print(age_audit.round(3).to_string(index=False))

BMI groups
          subgroup    n  prevalence   auc  sensitivity  specificity
 healthy (18.5-25)  555       0.260 0.691        0.229        0.912
       obese (30+) 1081       0.450 0.703        0.448        0.801
overweight (25-30)  684       0.357 0.730        0.381        0.873
     under (<18.5)  180       0.328 0.710        0.051        0.992
Age groups
subgroup   n  prevalence   auc  sensitivity  specificity
       1 179       0.151 0.588        0.037        0.980
      10 191       0.487 0.667        0.473        0.724
      11 194       0.521 0.656        0.535        0.656
      12 192       0.594 0.631        0.605        0.538
      13 203       0.576 0.619        0.658        0.442
       2 202       0.208 0.685        0.095        0.988
       3 187       0.225 0.681        0.024        0.986
       4 198       0.293 0.742        0.172        0.979
       5 201       0.298 0.662        0.167        0.950
       6 186       0.290 0.654        0.241        0.909
       7 20

In [8]:
def gap_summary(table, name):
    valid = table.dropna(subset=["sensitivity"])
    high = valid.loc[valid["sensitivity"].idxmax()]
    low = valid.loc[valid["sensitivity"].idxmin()]
    gap = high["sensitivity"] - low["sensitivity"]
    return {
        "comparison": name,
        "highest_sensitivity_group": high["subgroup"],
        "lowest_sensitivity_group": low["subgroup"],
        "sensitivity_gap": gap,
    }

gap_table = pd.DataFrame([
    gap_summary(bmi_audit, "BMI"),
    gap_summary(age_audit, "Age"),
])
gap_table.round(3)

  comparison  ... sensitivity_gap
0        BMI  ...           0.397
1        Age  ...           0.634

[2 rows x 4 columns]

**Plain-English conclusion:** A large
gap means the model catches true cases less consistently in one group than in
another. It should be disclosed and investigated. This table is a warning
signal, not proof that the model is fair or unfair, because subgroup size and
prevalence also affect the estimates.

## 7. Safety-reviewed lifestyle coach

The coach converts a risk estimate into general educational language. It never
diagnoses, prescribes medication, gives doses, or sets calorie targets. The live
app can use OpenAI when configured and otherwise uses this fixed safe template.

In [9]:
example = {"bmi": 31.2, "glucose": 145, "blood_pressure": 88, "age": 50}
print(llm_compose(example, risk=0.63))

Based on the inputs you shared, the model estimates your diabetes-risk band as **higher** (~63%).
Here are a few things worth a closer look:
- **Body Mass Index** — Your BMI is above 30, which is a category called obesity. Small, sustainable changes — like a daily 20-minute walk and swapping sugary drinks for water — are a great place to start.
- **Plasma glucose** — Your glucose reading is on the higher side. Ask your provider about an A1C test — it's a simple finger-stick that gives a 3-month average.
- **Blood pressure** — Your diastolic blood pressure is elevated. Tracking your BP weekly and watching sodium intake can help. Bring the log to your next checkup.
- **Age** — Adults 45+ are routinely screened for Type-2 diabetes. If you haven't had a screening lately, it's worth bringing up at your next visit.

Not medical advice. This message is for education only — please talk to a qualified healthcare provider before changing your diet, exercise, or medication.


## 8. Final limitations

- This model is not clinically validated and cannot diagnose diabetes.
- BRFSS is self-reported survey data; synthetic backup data is not patient data.
- PIMA represents only Pima women age 21 and older.
- Subgroup estimates can be unstable, especially for small groups.
- Feature explanations describe model behavior and do not establish cause.
- A 0.50 threshold is a demonstration default, not medical guidance.
- The app does not include every laboratory, clinical, or social factor.

**Not medical advice.** Anyone concerned about diabetes should speak with a
qualified healthcare provider.